In [2]:
from pathlib import Path
import datetime
import os
import random
import re
import shutil
import time

import dagshub
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import torch
from IPython.display import display
from roboflow import Roboflow
from ultralytics import YOLO, settings

In [3]:
rf = Roboflow(api_key="MQdx0fMQ8FiQPaS1VHRH")
project = rf.workspace("abiya-thesis").project("plant-pathology-2021-object-detection-j1jvh")
version = project.version(12)
dataset = version.download("yolov12", location="dataset")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to dataset in yolov12:: 100%|██████████| 21854/21854 [00:12<00:00, 1706.38it/s]


In [3]:
source_root = Path(getattr(globals().get("dataset", None), "location", "dataset"))
output_root = Path("split-dataset")

# dataset_sizes = [750, 1000, 1250, 1500, 1750, 2000, 3000, 4000, 5000]
dataset_sizes = [100, 200, 300, 400, 500, 600, 700]
seed = 42
class_names = ["frog-eye-leaf-spot", "healthy", "rust"]
img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def get_label_path(img_path: Path) -> Path | None:
    parts = list(img_path.parts)
    if "images" not in parts:
        return None
    idx = parts.index("images")
    parts[idx] = "labels"
    return Path(*parts).with_suffix(".txt")

def flatten_name(path: Path) -> str:
    return str(path.with_suffix("")).replace("\\", "__").replace("/", "__")

def write_split_yaml(split_dir: Path, names: list[str]) -> None:
    names_text = "[" + ", ".join(repr(name) for name in names) + "]"
    yaml_text = (
        f"path: {split_dir.resolve().as_posix()}\n"
        "train: train/images\n"
        "val: val/images\n"
        "test: test/images\n\n"
        f"nc: {len(names)}\n"
        f"names: {names_text}\n"
    )
    (split_dir / "data.yaml").write_text(yaml_text, encoding="utf-8")

# Kumpulkan pasangan image-label yang valid
pairs = []
for img_path in source_root.rglob("*"):
    if img_path.is_file() and img_path.suffix.lower() in img_exts:
        if "split-dataset" in img_path.parts:
            continue
        label_path = get_label_path(img_path)
        if label_path is not None and label_path.exists():
            pairs.append((img_path, label_path))

random.Random(seed).shuffle(pairs)

total_available = len(pairs)
print(f"Total pasangan image-label tersedia: {total_available}")

for split_size in dataset_sizes:
    if split_size > total_available:
        print(f"Ukuran {split_size} dilewati karena data hanya tersedia {total_available}.")
        continue

    train_count = int(split_size * 0.7)
    val_count = int(split_size * 0.1)

    subset = pairs[:split_size]
    train_pairs = subset[:train_count]
    val_pairs = subset[train_count:train_count + val_count]
    test_pairs = subset[train_count + val_count:]

    split_dir = output_root / str(split_size)
    for split_name in ["train", "val", "test"]:
        (split_dir / split_name / "images").mkdir(parents=True, exist_ok=True)
        (split_dir / split_name / "labels").mkdir(parents=True, exist_ok=True)

    def copy_pairs(pairs_list, split_name):
        for img_path, label_path in pairs_list:
            base_name = flatten_name(img_path.relative_to(source_root))
            dst_img = split_dir / split_name / "images" / f"{base_name}{img_path.suffix.lower()}"
            dst_lbl = split_dir / split_name / "labels" / f"{base_name}.txt"
            shutil.copy2(img_path, dst_img)
            shutil.copy2(label_path, dst_lbl)

    copy_pairs(train_pairs, "train")
    copy_pairs(val_pairs, "val")
    copy_pairs(test_pairs, "test")
    write_split_yaml(split_dir, class_names)

    print(
        f"Selesai {split_size}: train={len(train_pairs)}, val={len(val_pairs)}, test={len(test_pairs)} -> {split_dir}"
    )

Total pasangan image-label tersedia: 10921
Selesai 100: train=70, val=10, test=20 -> split-dataset\100
Selesai 200: train=140, val=20, test=40 -> split-dataset\200
Selesai 300: train=210, val=30, test=60 -> split-dataset\300
Selesai 400: train=280, val=40, test=80 -> split-dataset\400
Selesai 500: train=350, val=50, test=100 -> split-dataset\500
Selesai 600: train=420, val=60, test=120 -> split-dataset\600
Selesai 700: train=489, val=70, test=141 -> split-dataset\700


In [4]:
# ML Flow dan DagsHub Integration
dagshub.init(repo_owner='abiyamf', repo_name='thesis', mlflow=True)
settings.update({"mlflow": True})

Accessing as abiyamf

Initialized MLflow to track repo "abiyamf/thesis"

Repository abiyamf/thesis initialized!

In [5]:
os.environ["MLFLOW_KEEP_RUN_ACTIVE"] = "True"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

def format_duration(seconds: float) -> str:
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours}h {minutes}m {secs}s"

def save_class_metrics(val_results, mode):
    names = val_results.names
    data = {
        "class": [names[i] for i in range(len(names))],
        "precision": val_results.box.p.tolist(),
        "recall": val_results.box.r.tolist(),
        "mAP50": val_results.box.ap50.tolist(),
        "mAP50-95": val_results.box.maps.tolist(),
    }

    all_row = {
        "class": "all",
        "precision": float(val_results.box.mp),
        "recall": float(val_results.box.mr),
        "mAP50": float(val_results.box.map50),
        "mAP50-95": float(val_results.box.map),
    }

    for key in data.keys():
        data[key].insert(0, all_row[key])

    df = pd.DataFrame(data)
    csv_path = Path(val_results.save_dir) / f"class_metrics_{mode}.csv"
    df.to_csv(csv_path, index=False)

    return str(csv_path)

def write_validation_summary(result, csv_path: Path) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    csv_path.write_text(result.to_csv(), encoding="utf-8")

def read_map_metrics(csv_path: Path) -> tuple[float, float]:
    df = pd.read_csv(csv_path)
    if "class" in df.columns:
        class_col = "class"
    elif "Class" in df.columns:
        class_col = "Class"
    else:
        class_col = None

    if class_col is not None and (df[class_col].astype(str).str.lower() == "all").any():
        row = df[df[class_col].astype(str).str.lower() == "all"].iloc[0]
        return float(row["mAP50"]), float(row["mAP50-95"])

    # Fallback untuk file hasil result.to_csv() YOLO yang hanya berisi baris per class.
    return float(df["mAP50"].mean()), float(df["mAP50-95"].mean())

def build_summary_from_validation_files(project_base: Path) -> pd.DataFrame:
    summary_dir = project_base / "validation" / "summary"
    pattern = re.compile(r"(?P<model_size>.+)-split-(?P<split_size>\d+)-(?P<mode>train|test)-metrics\.csv$")
    rows = {}

    for csv_path in summary_dir.glob("*-metrics.csv"):
        match = pattern.match(csv_path.name)
        if match is None:
            continue

        model_size = match.group("model_size")
        split_size = int(match.group("split_size"))
        mode = match.group("mode")
        map50, map50_95 = read_map_metrics(csv_path)
        key = (model_size, split_size)
        rows.setdefault(key, {"model_size": model_size, "split_size": split_size})
        rows[key][f"{mode}_mAP50"] = map50
        rows[key][f"{mode}_mAP50_95"] = map50_95

    summary_df = pd.DataFrame(rows.values())
    if summary_df.empty:
        return summary_df

    summary_df = summary_df.dropna(subset=["train_mAP50", "test_mAP50"])
    return summary_df.sort_values(["model_size", "split_size"]).reset_index(drop=True)

def save_and_plot_summary(summary_df: pd.DataFrame, project_base: Path) -> None:
    if summary_df.empty:
        print("Belum ada metrik train/test yang bisa divisualisasikan.")
        return

    summary_csv = project_base / "training_summary_by_split.csv"
    summary_csv.parent.mkdir(parents=True, exist_ok=True)
    summary_df.to_csv(summary_csv, index=False)
    display(summary_df)

    plt.figure(figsize=(10, 6))
    for model_size, group in summary_df.groupby("model_size"):
        group = group.sort_values("split_size")
        plt.plot(group["split_size"], group["train_mAP50"], marker="o", label=f"Train {model_size}")
        plt.plot(group["split_size"], group["test_mAP50"], marker="s", linestyle="--", label=f"Test {model_size}")

    plt.title("Kenaikan Akurasi Train dan Test per Split Data")
    plt.xlabel("Jumlah data pada split")
    plt.ylabel("Akurasi (mAP50)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    accuracy_plot = project_base / "accuracy_by_split.png"
    plt.savefig(accuracy_plot, dpi=200)
    plt.show()

    print(f"Ringkasan disimpan ke: {summary_csv}")
    print(f"Plot akurasi disimpan ke: {accuracy_plot}")

In [8]:
# Variabel Global
projectType = "feedback_reviewer"
imgsz = 640
model_configs = [("yolo12n.pt", "nano")]
epochs = 50
batch = 16
data_path = Path("split-dataset")
project_base = Path("results/yolo_v12/feedback_reviewer")
available_splits = sorted(
    [p for p in data_path.iterdir() if p.is_dir() and (p / "data.yaml").exists()],
    key=lambda p: int(p.name),
)
available_splits = available_splits[:7]
print("Split yang akan dilatih:", [p.name for p in available_splits])

Split yang akan dilatih: ['100', '200', '300', '400', '500', '600', '700']


In [9]:
mlflow.set_experiment(f"thesis-{projectType}")
training_summaries = build_summary_from_validation_files(project_base).to_dict("records")
completed_splits = {
    (row["model_size"], int(row["split_size"]))
    for row in training_summaries
    if pd.notna(row.get("train_mAP50")) and pd.notna(row.get("test_mAP50"))
}

for model_name, model_size in model_configs:
    for split_dir in available_splits:
        split_size = int(split_dir.name)
        if (model_size, split_size) in completed_splits:
            print(f"Lewati {model_size} split {split_size}: metrik train/test sudah ada.")
            continue

        data_yaml = split_dir / "data.yaml"
        time_now = datetime.datetime.now().strftime("%m-%d_%H-%M-%S")
        run_name = f"{model_size}-split-{split_size}-{time_now}"
        model = YOLO(model_name)

        with mlflow.start_run(run_name=run_name):
            mlflow.log_params({
                "project_type": projectType,
                "model": model_name,
                "model_size": model_size,
                "split_size": split_size,
                "data_yaml": str(data_yaml),
                "gpu": gpu_name,
                "epochs": epochs,
                "batch": batch,
                "imgsz": imgsz,
            })

            start_time = time.time()
            training = model.train(
                data=str(data_yaml),
                epochs=epochs,
                imgsz=imgsz,
                batch=batch,
                project=str(project_base / "training" / model_size),
                name=f"split_{split_size}",
                exist_ok=True,
                plots=True,
            )
            elapsed = time.time() - start_time

            best_weights = Path(training.save_dir) / "weights" / "best.pt"
            trained_model = YOLO(str(best_weights if best_weights.exists() else model_name))

            train_eval = trained_model.val(
                data=str(data_yaml),
                imgsz=imgsz,
                project=str(project_base / "validation" / model_size),
                name=f"split_{split_size}_train",
                exist_ok=True,
                split="train",
            )
            test_eval = trained_model.val(
                data=str(data_yaml),
                imgsz=imgsz,
                project=str(project_base / "validation" / model_size),
                name=f"split_{split_size}_test",
                exist_ok=True,
                split="test",
            )

            summary_dir = project_base / "validation" / "summary"
            write_validation_summary(train_eval, summary_dir / f"{model_size}-split-{split_size}-train-metrics.csv")
            write_validation_summary(test_eval, summary_dir / f"{model_size}-split-{split_size}-test-metrics.csv")

            train_class_metrics = save_class_metrics(train_eval, "train")
            test_class_metrics = save_class_metrics(test_eval, "test")
            mlflow.log_artifact(train_class_metrics, artifact_path="class_metrics")
            mlflow.log_artifact(test_class_metrics, artifact_path="class_metrics")

            metrics = {
                "train_mAP50": float(train_eval.box.map50),
                "train_mAP50_95": float(train_eval.box.map),
                "test_mAP50": float(test_eval.box.map50),
                "test_mAP50_95": float(test_eval.box.map),
            }
            mlflow.log_metrics(metrics)
            mlflow.log_param("training_time", format_duration(elapsed))

            print(
                f"Selesai {model_size} split {split_size}: "
                f"train mAP50={metrics['train_mAP50']:.4f}, test mAP50={metrics['test_mAP50']:.4f}"
            )

        # Simpan ulang summary dan plot setelah setiap split selesai.
        summary_df = build_summary_from_validation_files(project_base)
        save_and_plot_summary(summary_df, project_base)
        completed_splits.add((model_size, split_size))

New https://pypi.org/project/ultralytics/8.4.46 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.0  Python-3.12.12 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=split-dataset\100\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=split_100, nbs=6

,model_size,split_size,test_mAP50,test_mAP50_95,train_mAP50,train_mAP50_95
0,nano,100,0.576887,0.540873,0.632073,0.618650
1,nano,750,0.930803,0.851967,0.986920,0.965513
2,nano,1000,0.905773,0.839843,0.986080,0.963213
3,nano,1250,0.957177,0.901227,0.989087,0.966697
4,nano,1500,0.964817,0.918633,0.985893,0.964907
5,nano,1750,0.942217,0.894710,0.988663,0.967827
6,nano,2000,0.940753,0.896000,0.988143,0.967657
7,nano,3000,0.960183,0.914897,0.986400,0.964483
8,nano,4000,0.958583,0.916713,0.986573,0.966497


<Figure size 1000x600 with 1 Axes>

Ringkasan disimpan ke: results\yolo_v12\feedback_reviewer\training_summary_by_split.csv
Plot akurasi disimpan ke: results\yolo_v12\feedback_reviewer\accuracy_by_split.png
New https://pypi.org/project/ultralytics/8.4.46 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.0  Python-3.12.12 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=split-dataset\200\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, 

,model_size,split_size,test_mAP50,test_mAP50_95,train_mAP50,train_mAP50_95
0,nano,100,0.576887,0.540873,0.632073,0.618650
1,nano,200,0.760120,0.728120,0.979800,0.963590
2,nano,750,0.930803,0.851967,0.986920,0.965513
3,nano,1000,0.905773,0.839843,0.986080,0.963213
4,nano,1250,0.957177,0.901227,0.989087,0.966697
5,nano,1500,0.964817,0.918633,0.985893,0.964907
6,nano,1750,0.942217,0.894710,0.988663,0.967827
7,nano,2000,0.940753,0.896000,0.988143,0.967657
8,nano,3000,0.960183,0.914897,0.986400,0.964483
9,nano,4000,0.958583,0.916713,0.986573,0.966497


...

<Figure size 1000x600 with 1 Axes>

Ringkasan disimpan ke: results\yolo_v12\feedback_reviewer\training_summary_by_split.csv
Plot akurasi disimpan ke: results\yolo_v12\feedback_reviewer\accuracy_by_split.png
New https://pypi.org/project/ultralytics/8.4.46 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.0  Python-3.12.12 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=split-dataset\700\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, 

,model_size,split_size,test_mAP50,test_mAP50_95,train_mAP50,train_mAP50_95
0,nano,100,0.576887,0.540873,0.632073,0.618650
1,nano,200,0.760120,0.728120,0.979800,0.963590
2,nano,300,0.812700,0.747497,0.994463,0.988593
3,nano,400,0.777840,0.714320,0.971670,0.938523
4,nano,500,0.886377,0.853100,0.994750,0.985200
5,nano,600,0.926467,0.885897,0.994940,0.987093
6,nano,700,0.871577,0.811190,0.994790,0.982183
7,nano,750,0.930803,0.851967,0.986920,0.965513
8,nano,1000,0.905773,0.839843,0.986080,0.963213
9,nano,1250,0.957177,0.901227,0.989087,0.966697


<Figure size 1000x600 with 1 Axes>

Ringkasan disimpan ke: results\yolo_v12\feedback_reviewer\training_summary_by_split.csv
Plot akurasi disimpan ke: results\yolo_v12\feedback_reviewer\accuracy_by_split.png


In [10]:
# Jalankan cell ini kapan pun setelah notebook berhenti/restart untuk membuat ulang visualisasi
# dari semua file metrik split yang sudah selesai.
summary_df = build_summary_from_validation_files(project_base)
save_and_plot_summary(summary_df, project_base)

,model_size,split_size,test_mAP50,test_mAP50_95,train_mAP50,train_mAP50_95
0,nano,100,0.576887,0.540873,0.632073,0.618650
1,nano,200,0.760120,0.728120,0.979800,0.963590
2,nano,300,0.812700,0.747497,0.994463,0.988593
3,nano,400,0.777840,0.714320,0.971670,0.938523
4,nano,500,0.886377,0.853100,0.994750,0.985200
5,nano,600,0.926467,0.885897,0.994940,0.987093
6,nano,700,0.871577,0.811190,0.994790,0.982183
7,nano,750,0.930803,0.851967,0.986920,0.965513
8,nano,1000,0.905773,0.839843,0.986080,0.963213
9,nano,1250,0.957177,0.901227,0.989087,0.966697


<Figure size 1000x600 with 1 Axes>

Ringkasan disimpan ke: results\yolo_v12\feedback_reviewer\training_summary_by_split.csv
Plot akurasi disimpan ke: results\yolo_v12\feedback_reviewer\accuracy_by_split.png
